# CSC699 LoRA Fine-Tuning v2: RAG-Aware Adapter

This notebook retrains the Mistral-7B LoRA adapter with a **RAG-aware prompt template**.

### What changed from v1 (`CSC699LoRA.ipynb`):
1. `create_prompt()` now includes a `### Reference Examples from Database:` section
2. ~20% of training examples get synthetic RAG context so the model learns to use it
3. The remaining ~80% get the placeholder `No verified historical examples were found.`
4. The adapter output path is `final_adapter_v2` to avoid overwriting v1

### Why:
The v1 adapter was trained without the RAG section. When inference injected
`### Reference Examples from Database:` into the prompt, the model had never seen
that section header during training, causing hallucination (Synenga Malware, Algobila,
fake MITRE IDs). This v2 adapter learns the full prompt structure.

### Base model: `mistralai/Mistral-7B-Instruct-v0.1`
Must match `LOCAL_BASE_MODEL` in `config.py` (already fixed to v0.1).

## Install Dependencies

In [ ]:
!pip install -q -U torch transformers datasets accelerate bitsandbytes peft trl

## Authenticate & Mount Drive

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

from google.colab import drive
drive.mount('/content/drive')

## Load Dataset, Model & Configure LoRA

Identical to v1: same base model, same quantization, same LoRA rank/alpha.

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Load the Dataset
print("Loading dataset...")
ds = load_dataset("tumeteor/Security-TTP-Mapping")

# 2. Configure 4-bit Quantization (NF4)
nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 3. Load Model and Tokenizer -- MUST be v0.1 to match config.py
model_id = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=nf4_config,
    device_map="auto",
    trust_remote_code=True
)

# 4. Prepare Model for LoRA
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

print(f"Trainable parameters: {model.print_trainable_parameters()}")

## RAG-Aware Prompt Template (THE KEY CHANGE)

This is the critical difference from v1. The prompt now always includes:
```
### Reference Examples from Database:
```

- ~20% of examples get a **synthetic RAG hit** (reuses the same log text as a fake retrieval result)
- ~80% of examples get `No verified historical examples were found.`

This teaches the model to:
1. Recognize and ignore the RAG section when empty
2. Use RAG context as supporting evidence when present
3. Always output `['T1234']` format regardless of RAG presence

In [ ]:
import random
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

# Rename the conflicting column (same fix as v1)
ds = ds.rename_column("labels", "mitre_tags")

# Seed for reproducibility during training
random.seed(42)

def create_prompt(example):
    bos_token = "<s>"
    eos_token = "</s>"
    system_message = (
        "You are a cybersecurity assistant. "
        "Your task is to analyze a system log and assign the most "
        "appropriate MITRE ATT&CK technique(s)."
    )
    log_text = example["text1"]
    mitre_tags = "".join(example["mitre_tags"]).strip()

    # 20% of training examples include a synthetic RAG retrieval hit.
    # This teaches the model to use the ### Reference Examples section.
    if random.random() < 0.2:
        # Simulate a RAG result using the same log as a "historical" match
        snippet = log_text[:120].replace('\n', ' ').strip()
        db_results = (
            f"1. Summary: {snippet}...\n"
            f"   Verified MITRE IDs: {mitre_tags} (similarity=0.850)"
        )
    else:
        db_results = "No verified historical examples were found."

    full_prompt = (
        f"{bos_token}\n"
        f"### Instruction:\n{system_message}\n\n"
        f"### Reference Examples from Database:\n{db_results}\n\n"
        f"### Log:\n{log_text}\n\n"
        f"### Response:\n{mitre_tags}\n"
        f"{eos_token}"
    )
    return full_prompt


# Verify the prompt looks correct
print("=== Example prompt (no RAG) ===")
random.seed(0)  # Force no-RAG path for demo
print(create_prompt(ds['train'][0]))
print()
print("=== Example prompt (with RAG) ===")
random.seed(5)  # Force RAG path for demo
print(create_prompt(ds['train'][0]))

# Reset seed for actual training
random.seed(42)

## Training Configuration & Execution

Same hyperparameters as v1. Output goes to `final_adapter_v2` to preserve the original.

In [ ]:
# Training Arguments (identical to v1 except output path)
args = SFTConfig(
    output_dir="/content/drive/MyDrive/CSC699/HF/siem-finetuned-v2",
    max_length=1024,
    max_steps=2400,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=100,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=5,
    learning_rate=1e-4,
    bf16=True,
    lr_scheduler_type="constant",
    metric_for_best_model="eval_loss",
    report_to=[]
)

# Initialize Trainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    formatting_func=create_prompt,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Execute Training
print("Starting RAG-aware training (v2)...")
trainer.train()

# Save the final model adapter to a NEW path (preserves v1)
ADAPTER_PATH_V2 = "/content/drive/MyDrive/CSC699/HF/final_adapter_v2"
trainer.model.save_pretrained(ADAPTER_PATH_V2)
tokenizer.save_pretrained(ADAPTER_PATH_V2)
print(f"Training complete! RAG-aware adapter saved to: {ADAPTER_PATH_V2}")

## Quick Smoke Test

Verify the new adapter produces clean output on a sample prompt.

In [ ]:
from peft import PeftModel

# Reload fresh base + new adapter for inference test
print("Loading base model for smoke test...")
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.1",
    quantization_config=nf4_config,
    device_map="auto",
    trust_remote_code=True
)

adapter_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH_V2)

# Test 1: Without RAG context
test_prompt_no_rag = """<s>
### Instruction:
You are a cybersecurity assistant. Your task is to analyze a system log and assign the most appropriate MITRE ATT&CK technique(s).

### Reference Examples from Database:
No verified historical examples were found.

### Log:
A connection from 10.128.1.13 to 192.168.220.132 on port 3389 (RDP) indicates suspicious network activity that may be associated with attacks or unauthorized access.

### Response:
"""

# Test 2: With RAG context
test_prompt_with_rag = """<s>
### Instruction:
You are a cybersecurity assistant. Your task is to analyze a system log and assign the most appropriate MITRE ATT&CK technique(s).

### Reference Examples from Database:
1. Summary: RDP session hijacking detected on internal network segment via tscon.exe
   Verified MITRE IDs: ['T1563.002'] (similarity=0.912)

### Log:
A connection from 10.128.1.13 to 192.168.220.132 on port 3389 (RDP) indicates suspicious network activity that may be associated with attacks or unauthorized access.

### Response:
"""

for label, prompt in [("No RAG", test_prompt_no_rag), ("With RAG", test_prompt_with_rag)]:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        output = adapter_model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][len(inputs["input_ids"][0]):]
    result = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    print(f"\n=== {label} ===")
    print(f"Output: {result}")
    # Truncate at first ] for clean display
    bracket = result.find(']')
    if bracket != -1:
        print(f"Clean:  {result[:bracket+1]}")

print("\nSmoke test complete!")
print(f"\nNext step: Download the adapter from Google Drive at:")
print(f"  {ADAPTER_PATH_V2}")
print(f"Copy it to your local machine at:")
print(f"  C:\\Users\\WillYoung\\Downloads\\CSC699\\final_adapter_v2")
print(f"Then update LOCAL_ADAPTER_PATH in config.py to point to the v2 adapter.")